## Benchmark Run Script (Reference)

```bash
cd /Users/RyanLandvater/Programming_Projects/FHIR_Testing
DYLD_LIBRARY_PATH=local/lib ./build/bench/bench/bench_harness --warmup-iterations 2 --iterations 10 --db "host=localhost port=5432 dbname=benchmark user=bench password=bench"
```

Optional explicit runs (same as current default):

```bash
cd /Users/RyanLandvater/Programming_Projects/FHIR_Testing
DYLD_LIBRARY_PATH=local/lib ./build/bench/bench/bench_harness --warmup-iterations 2 --iterations 10 --db "host=localhost port=5432 dbname=benchmark user=bench password=bench" --runs 10
```

DYLD_LIBRARY_PATH=local/lib ./build/bench/bench/bench_harness --warmup-iterations 2 --iterations 10 --db "host=localhost port=5432 dbname=fhir_benchmark user=postgres password=postgres"


# FastFHIR vs JSON-FHIR Benchmark Results

This notebook loads and analyzes benchmark results from the C++ harness.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
from sqlalchemy import create_engine, text

# Configuration - read from environment or use defaults
db_host = os.getenv('POSTGRES_HOST', 'localhost')
db_port = os.getenv('POSTGRES_PORT', '5432')
db_name = os.getenv('POSTGRES_DB', 'fhir_benchmark')
db_user = os.getenv('POSTGRES_USER', 'postgres')
db_pass = os.getenv('POSTGRES_PASSWORD', 'postgres')

db_url = f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}"
print(f"Connecting to {db_user}@{db_host}:{db_port}/{db_name}")

## Load Results from Database

In [ ]:
try:
    engine = create_engine(db_url, pool_pre_ping=True)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Connected successfully (SQLAlchemy)")
except Exception as e:
    print(f"Connection failed: {e}")
    engine = None

## Summary Statistics

In [ ]:
if 'engine' in locals() and engine is not None:
    query = """
    SELECT 
        br.id,
        run_id,
        arm,
        stage,
        duration_us,
        target_mb,
        patients_in_bundle,
        br.created_at
    FROM benchmark_results br
    ORDER BY run_id DESC, target_mb, arm, stage
    LIMIT 10000
    """
    
    # Store full result set for analysis (no automatic printing).
    df = pd.read_sql(query, engine)
    
    # Convenience DataFrames for manual inspection when needed.
    df_preview = df.head(20).copy()
    latest_run_id = int(df['run_id'].max()) if len(df) > 0 else None
    latest_run_df = df[df['run_id'] == latest_run_id].copy() if latest_run_id is not None else pd.DataFrame()
else:
    df = pd.DataFrame()
    df_preview = pd.DataFrame()
    latest_run_df = pd.DataFrame()
    latest_run_id = None

## Summary Statistics

In [ ]:
if 'df' in locals() and len(df) > 0:
    summary = df.groupby(['arm', 'stage'])['duration_us'].agg(['mean', 'std', 'min', 'max']).round(2)
    arm_totals = df.groupby('arm')['duration_us'].agg(['sum', 'mean', 'count']).round(2)
    
    # Keep optional preview frames ready for manual review.
    summary_preview = summary.reset_index().copy()
    arm_totals_preview = arm_totals.reset_index().copy()
else:
    summary = pd.DataFrame()
    arm_totals = pd.DataFrame()
    summary_preview = pd.DataFrame()
    arm_totals_preview = pd.DataFrame()
summary_preview

## Performance Comparison - Serialization

In [ ]:
if 'df' in locals() and len(df) > 0:
    # Use all runs for stable CI estimates while still plotting one point per arm/target.
    use_latest_run_only = False
    source_df = latest_run_df if use_latest_run_only and 'latest_run_df' in locals() and len(latest_run_df) > 0 else df
    stage1_data = source_df[source_df['stage'] == 'stage1_serialize'].copy()

    if len(stage1_data) > 0:
        show_smoke_arms = False
        if not show_smoke_arms:
            stage1_data = stage1_data[stage1_data['arm'].isin(['fastfhir', 'json_fhir'])].copy()

        fig, ax = plt.subplots(figsize=(12, 6))

        grouped = (
            stage1_data
            .groupby(['arm', 'target_mb'])['duration_us']
            .agg(mean='mean', std='std', count='count', min='min', max='max')
            .reset_index()
        )

        for arm in grouped['arm'].unique():
            data = grouped[grouped['arm'] == arm].sort_values('target_mb').copy()
            y = data['mean'].to_numpy()

            # Existing min/max error bars.
            lower = (data['mean'] - data['min']).to_numpy()
            upper = (data['max'] - data['mean']).to_numpy()
            yerr = np.vstack([lower, upper])

            # 95% confidence interval shading around the mean.
            sem = (data['std'] / np.sqrt(data['count'].clip(lower=1))).fillna(0.0)
            ci95 = 1.96 * sem
            ci_low = np.maximum((data['mean'] - ci95).to_numpy(), 1e-9)
            ci_high = (data['mean'] + ci95).to_numpy()

            eb = ax.errorbar(
                data['target_mb'],
                y,
                yerr=yerr,
                marker='o',
                capsize=4,
                linewidth=2,
                label=arm,
            )
            line_color = eb.lines[0].get_color()
            ax.fill_between(
                data['target_mb'],
                ci_low,
                ci_high,
                color=line_color,
                alpha=0.18,
                linewidth=0,
            )

        ax.set_xlabel('Bundle Size (MB)', fontsize=12)
        ax.set_ylabel('Serialization Time (microseconds)', fontsize=12)
        ax.set_title('Serialization Performance by Arm (mean with min/max + 95% CI)', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')
        plt.tight_layout()
        plt.show()
    else:
        print('No serialization data found')

## Performance Comparison - Query

In [ ]:
if 'df' in locals() and len(df) > 0:
    # Use all runs for stable CI estimates while still plotting one point per arm/target.
    use_latest_run_only = False
    source_df = latest_run_df if use_latest_run_only and 'latest_run_df' in locals() and len(latest_run_df) > 0 else df
    stage3_data = source_df[source_df['stage'] == 'stage3_query'].copy()

    if len(stage3_data) > 0:
        show_smoke_arms = False
        if not show_smoke_arms:
            stage3_data = stage3_data[stage3_data['arm'].isin(['fastfhir', 'json_fhir'])].copy()

        fig, ax = plt.subplots(figsize=(12, 6))

        grouped = (
            stage3_data
            .groupby(['arm', 'target_mb'])['duration_us']
            .agg(mean='mean', std='std', count='count', min='min', max='max')
            .reset_index()
        )

        for arm in grouped['arm'].unique():
            data = grouped[grouped['arm'] == arm].sort_values('target_mb').copy()
            y = data['mean'].to_numpy()

            # Existing min/max error bars.
            lower = (data['mean'] - data['min']).to_numpy()
            upper = (data['max'] - data['mean']).to_numpy()
            yerr = np.vstack([lower, upper])

            # 95% confidence interval shading around the mean.
            sem = (data['std'] / np.sqrt(data['count'].clip(lower=1))).fillna(0.0)
            ci95 = 1.96 * sem
            ci_low = np.maximum((data['mean'] - ci95).to_numpy(), 1e-9)
            ci_high = (data['mean'] + ci95).to_numpy()

            eb = ax.errorbar(
                data['target_mb'],
                y,
                yerr=yerr,
                marker='s',
                capsize=4,
                linewidth=2,
                label=arm,
            )
            line_color = eb.lines[0].get_color()
            ax.fill_between(
                data['target_mb'],
                ci_low,
                ci_high,
                color=line_color,
                alpha=0.18,
                linewidth=0,
            )

        ax.set_xlabel('Bundle Size (MB)', fontsize=12)
        ax.set_ylabel('Query Time (microseconds)', fontsize=12)
        ax.set_title('Query Performance by Arm (mean with min/max + 95% CI)', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.set_yscale('log')
        plt.tight_layout()
        plt.show()
    else:
        print('No query data found')

## Cleanup

In [ ]:
if 'engine' in locals() and engine is not None:
    engine.dispose()
    print("Database engine disposed")